In [ ]:
# baselines.ipynb
"""
Four baselines to compare against the MTL model:
# 1. logistic regression (demo + disease flags)
# 2. random forest (demo + disease flags)
# 3. STL neural network (demo + disease flags, no shared trunk)
# 4. MTL no diseases (demo only, shared trunk, no disease flag inputs)
"""

import numpy as np
import pandas as pd
import json
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

In [ ]:
# Load everything the same way as always
data = pd.read_csv('saved/nhanes_clean.csv')
with open('saved/col_lists.json', 'r') as f:
    cols = json.load(f)
with open('saved/demo_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

demo_cols    = [c for c in cols['demo_cols'] if c not in ['SLQ300', 'SLQ310']]
disease_cols = cols['disease_cols']
disease_meta = cols['disease_meta']

continuous_cols = ['RIDAGEYR','BMXBMI','BMXWAIST','ALQ121','PAQ706',
                   'OCQ180','HUQ010','SBP','DBP']

# transform only, never refit
data[continuous_cols] = scaler.transform(data[continuous_cols])

# same split as always
train_df, temp_df = train_test_split(data, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
K = len(disease_cols)
n_demo = len(demo_cols)